# 07 — Seaborn: Statistical Visualization
**Goal:** Use Seaborn's figure-level functions to produce statistical charts
with minimal code, and to understand what they build under the hood.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print('seaborn', sns.__version__)

## 1. The two-level API

Seaborn is organized as:

```
Figure-level      relplot, displot, catplot, jointplot, pairplot
  → one Figure, optional faceting by col/row
Axes-level        scatterplot, histplot, boxplot, violinplot, ...
  → one Axes, drop into existing figure
```

**Rule of thumb:**

- Use **Axes-level** when you are building a custom multi-panel figure.
- Use **Figure-level** when the chart *is* the figure.

In [ ]:
df = pd.DataFrame({
    'channel': np.random.choice(['Search', 'Social', 'Display', 'Email'], 400),
    'spend'  : np.random.gamma(2, 50, 400),
    'conv'   : np.random.gamma(1, 8, 400),
    'region' : np.random.choice(['EMEA', 'APAC', 'AMER'], 400),
})
df.head()

## 2. `relplot` — relational plots (scatter + line)

`relplot` is the figure-level entry point for showing the relationship between
two variables. `kind='scatter'` is default; `kind='line'` connects points.

In [ ]:
g = sns.relplot(data=df, x='spend', y='conv', hue='channel',
                kind='scatter', height=5, aspect=1.3)
g.set(title='Spend vs conversions, by channel'); plt.show()

In [ ]:
g = sns.relplot(data=df, x='spend', y='conv', hue='channel',
                col='region', kind='scatter', height=4, aspect=0.9)
g.fig.suptitle('Faceting — one panel per region', y=1.03)
plt.show()

## 3. `displot` — distributions

`kind` controls the geometry: `hist`, `kde`, `ecdf`.

In [ ]:
g = sns.displot(data=df, x='spend', hue='channel', kind='kde',
                height=5, aspect=1.3, fill=True, alpha=0.4)
g.set(title='KDE of spend by channel'); plt.show()

## 4. `catplot` — categorical comparisons

`kind` choices: `strip`, `swarm`, `box`, `violin`, `boxen`, `bar`, `count`,
`point`.

In [ ]:
g = sns.catplot(data=df, x='channel', y='conv', kind='box',
                height=5, aspect=1.3)
g.set(title='Box — conv by channel'); plt.show()

In [ ]:
g = sns.catplot(data=df, x='channel', y='conv', kind='violin',
                hue='region', split=True, height=5, aspect=1.3, inner='quartile')
g.set(title='Violin — conv by channel split by region'); plt.show()

## 5. `jointplot` — bivariate + marginals in one

`jointplot` puts a bivariate chart in the center and the marginals of each
variable on the edges.

In [ ]:
g = sns.jointplot(data=df, x='spend', y='conv', kind='hex', height=6,
                  marginal_kws=dict(bins=30, fill=True))
g.fig.suptitle('Joint — hexbin + marginals', y=1.02); plt.show()

## 6. `pairplot` — the screen for many variables

Pairplot draws every numeric pair as a scatter (or KDE) plus the diagonal
histogram. It is a *first-look* tool, not a final report.

In [ ]:
g = sns.pairplot(df, hue='channel', diag_kind='kde', height=2.2,
                 plot_kws=dict(alpha=0.5, s=20))
g.fig.suptitle('Pairplot — every variable pair', y=1.02); plt.show()

## 7. The heatmap family

Pivot to a matrix, then `sns.heatmap`. For correlation matrices, also use
`sns.clustermap` to surface the natural grouping of variables.

In [ ]:
pivot = df.pivot_table(index='region', columns='channel', values='conv', aggfunc='mean')
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlGnBu', ax=ax)
ax.set_title('Mean conversions — region × channel')
plt.tight_layout(); plt.show()

In [ ]:
num = df.select_dtypes('number')
g = sns.clustermap(num.corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0)
g.fig.suptitle('Clustermap of correlations', y=1.02); plt.show()

## 8. Aesthetic control

Seaborn has its own theming layer on top of matplotlib.

- `sns.set_theme(style=..., palette=...)` sets the global style.
- `sns.set_style('whitegrid' | 'darkgrid' | 'white' | 'dark' | 'ticks')`
- `sns.set_context('paper' | 'notebook' | 'talk' | 'poster')` — relative font
  sizes for the target medium.
    
    Use `with sns.axes_style(...)` to scope a style to one block.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, ctx in zip(axes, ['paper', 'notebook', 'talk']):
    with sns.axes_style('whitegrid'):
        sns.boxplot(data=df, x='channel', y='conv', ax=ax)
    ax.set_title(f'context = {ctx}')
plt.tight_layout(); plt.show()

## 9. Mixing seaborn and matplotlib

Seaborn is *built on* matplotlib. You can:

- Pass an existing `ax=ax` to drop a seaborn chart into a panel you control.
- Call `plt.*` after a seaborn figure-level call to add titles, references,
  annotations on top.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
sns.violinplot(data=df, x='channel', y='conv', ax=axes[0], color='steelblue')
sns.boxplot  (data=df, x='channel', y='conv', ax=axes[1], color='lightblue')
axes[0].set_title('Violin'); axes[1].set_title('Box')
for ax in axes: ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

## 10. Real data

Apply the family to the unified marketing table.

In [ ]:
mkt = pd.read_csv('data/clean/unified_daily.csv')
print(mkt.columns.tolist()[:10])
mkt.head(2)

In [ ]:
num_cols = mkt.select_dtypes('number').columns.tolist()[:3]
if len(num_cols) >= 2:
    g = sns.jointplot(data=mkt, x=num_cols[0], y=num_cols[1],
                      kind='reg', height=6, marginal_kws=dict(bins=30, fill=True))
    g.fig.suptitle(f'{num_cols[0]} vs {num_cols[1]}', y=1.02)
    plt.show()

## Summary

| Function | Use |
|---|---|
| `relplot` | Relationship between two variables + facets |
| `displot` | Distribution (hist, kde, ecdf) |
| `catplot` | Categorical comparison |
| `jointplot` | Bivariate + marginals |
| `pairplot` | Many variables, fast screening |
| `heatmap` / `clustermap` | Matrices |
| Axes-level | `scatterplot`, `boxplot`, `kdeplot`, `histplot`, ... |

**Next:** `08_plotly_interactive.ipynb` — interactivity, hover, animations.